<a href="https://colab.research.google.com/github/TamaraCucumides/MDS3020/blob/main/Clase4_autoML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demo AutoML + LightGBM + SHAP (Colab)

Este notebook es una demo corta y moderna para tabular data usando el dataset Breast Cancer de sklearn.

**Incluye:**

- AutoML ligero con **AutoGluon** (selección automática de modelos e hiperparámetros)
- AutoML alternativo con **H2O AutoML**
- Baseline moderno para tabular data con **LightGBM**
- Interpretabilidad rápida con **SHAP** sobre el baseline LightGBM


## 1. Instalación

> En Colab usa `%pip` para instalar en el kernel correcto.

In [ ]:
%pip install -U autogluon h2o lightgbm shap scikit-learn pandas --quiet

## 2. Cargar datos y preparar split

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer(as_frame=True)
df = data.frame.copy()

target_col = data.target.name  # <-- "target"

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df[target_col]
)

train_df.head()


## 3. AutoML ligero con AutoGluon

In [ ]:
from autogluon.tabular import TabularPredictor

predictor_ag = TabularPredictor(
    label=target_col,
    eval_metric="roc_auc"
).fit(
    train_df,
    time_limit=60,             # AutoML ligero: 60s
    presets="medium_quality"
)

# Leaderboard de modelos probados
lb_ag = predictor_ag.leaderboard(test_df, silent=True)
lb_ag.head(10)

## 4. Evaluación AutoGluon

In [ ]:
perf_ag = predictor_ag.evaluate(test_df, silent=True)
perf_ag

## 5. Baseline moderno: LightGBM

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_test  = test_df.drop(columns=[target_col])
y_test  = test_df[target_col]

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

lgbm.fit(X_train, y_train, )

pred_proba_lgbm = lgbm.predict_proba(X_test)[:, 1]
auc_lgbm = roc_auc_score(y_test, pred_proba_lgbm)

print("LightGBM ROC-AUC:", auc_lgbm)

## 6. Interpretabilidad rápida con SHAP (LightGBM)

Importancia global.

In [ ]:
import shap

shap.initjs()

explainer = shap.TreeExplainer(lgbm)
shap_values = explainer(X_test)

# Importancia global (bar plot)
shap.summary_plot(shap_values, X_test, plot_type="bar")

## 7. SHAP summary plot (beeswarm)

Dirección e impacto de cada variable.

In [ ]:
shap.summary_plot(shap_values, X_test)

## 8. AutoML alternativo con H2O AutoML

In [ ]:
import h2o
from h2o.automl import H2OAutoML

h2o.init()

train_h2o = h2o.H2OFrame(train_df)
test_h2o  = h2o.H2OFrame(test_df)

# Clasificación binaria
train_h2o[target_col] = train_h2o[target_col].asfactor()
test_h2o[target_col]  = test_h2o[target_col].asfactor()

aml = H2OAutoML(
    max_runtime_secs=60,  # AutoML ligero 60s
    seed=42,
    sort_metric="AUC"
)
aml.train(y=target_col, training_frame=train_h2o)

aml.leaderboard.head(10)

## 9. Evaluación H2O

In [ ]:
perf_h2o = aml.leader.model_performance(test_h2o)
print("H2O Leader AUC:", perf_h2o.auc())
print("Mejor modelo H2O:", aml.leader.model_id)

## 10. Comparación final

In [ ]:
print("=== RESUMEN ===")
print("AutoGluon ROC-AUC:", perf_ag["roc_auc"])
print("LightGBM ROC-AUC:", auc_lgbm)
print("H2O Leader AUC   :", perf_h2o.auc())